In [1]:
import json
import sys
import numpy as np

sys.path.insert(0, '..')
from transformers import AutoTokenizer

TOKENIZER = 'Qwen/Qwen2.5-1.5B'
MAX_NEW_TOKENS = 512  # threshold for truncation

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
print('Tokenizer loaded')

/home/r.dyachenko/miniconda3/envs/llm-alignment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded


In [2]:
def stats(path, max_new_tokens=MAX_NEW_TOKENS):
    """Compute avg_tokens (real, from response text) and truncated% for a results file."""
    with open(path) as f:
        data = json.load(f)
    lengths = [len(tokenizer.encode(r['response'], add_special_tokens=False)) for r in data]
    truncated = sum(1 for l in lengths if l >= max_new_tokens - 10)
    correct = sum(1 for r in data if r['correct'])
    return {
        'n': len(data),
        'accuracy': correct / len(data),
        'avg_tokens': round(np.mean(lengths)),
        'truncated_n': truncated,
        'truncated_pct': truncated / len(data) * 100,
    }

def stats_sc(path):
    """SC results have no response field — accuracy only."""
    with open(path) as f:
        data = json.load(f)
    correct = sum(1 for r in data if r['correct'])
    return {'n': len(data), 'accuracy': correct / len(data), 'avg_tokens': None, 'truncated_pct': None}

In [3]:
ROWS = [
    ('Zero-shot (Base)',           '../results/gsm8k_baseline/base_zeroshot_test3_results.json'),
    ('CoT (Base)',                 '../results/gsm8k_baseline/base_cot_test3_results.json'),
    ('LoRA SFT (checkpoint-1317)', '../results/gsm8k_sft/checkpoint-1317_results.json'),
    ('SFT + DAPO (ckpt-2760)',     '../results/gsm8k_grpo/dapo/dapo_checkpoint-2760_results.json'),
    ('SFT + Dr.GRPO (ckpt-1840)',  '../results/gsm8k_grpo/dr_grpo/dr_grpo_checkpoint-1840_results.json'),
    ('Qwen2.5-1.5B-Instruct',     '../results/gsm8k_instruct/Qwen2.5-1.5B-Instruct_cot_results.json'),
    ('Qwen2.5-Math-Instruct',     '../results/gsm8k_instruct/Qwen2.5-Math-1.5B-Instruct_cot_results.json'),
]

results = {}
for label, path in ROWS:
    s = stats(path)
    results[label] = s
    print(f"{label:<42} acc={s['accuracy']*100:.1f}%  avg_tokens={s['avg_tokens']:>4}  truncated={s['truncated_pct']:>5.1f}%")

sc = stats_sc('../results/gsm8k_sc/checkpoint-2760_sc8_results.json')
results['SFT + DAPO + SC@8'] = sc
print(f"{'SFT + DAPO + SC@8':<42} acc={sc['accuracy']*100:.1f}%  avg_tokens=—  truncated=—")

Zero-shot (Base)                           acc=65.3%  avg_tokens= 268  truncated=  5.5%
CoT (Base)                                 acc=70.6%  avg_tokens= 274  truncated=  4.9%
LoRA SFT (checkpoint-1317)                 acc=71.3%  avg_tokens= 127  truncated=  0.5%
SFT + DAPO (ckpt-2760)                     acc=78.2%  avg_tokens= 138  truncated=  0.4%
SFT + Dr.GRPO (ckpt-1840)                  acc=77.9%  avg_tokens= 139  truncated=  0.4%
Qwen2.5-1.5B-Instruct                      acc=68.5%  avg_tokens= 308  truncated=  4.1%
Qwen2.5-Math-Instruct                      acc=85.7%  avg_tokens= 301  truncated=  4.1%
SFT + DAPO + SC@8                          acc=79.7%  avg_tokens=—  truncated=—


In [4]:
def row_md(label, s, bold=False):
    acc   = f"{s['accuracy']*100:.1f}%"
    tok   = str(s['avg_tokens'])          if s['avg_tokens']    is not None else '—'
    trunc = f"{s['truncated_pct']:.1f}%" if s['truncated_pct'] is not None else '—'
    if bold:
        return f'| **{label}** | **{acc}** | **{tok}** | **{trunc}** |'
    return f'| {label} | {acc} | {tok} | {trunc} |'

print('| Method | GSM8K Accuracy | Avg Tokens | Truncated |')
print('|--------|:-:|:-:|:-:|')
print(row_md('Zero-shot (Base)',                      results['Zero-shot (Base)']))
print(row_md('CoT Prompting',                         results['CoT (Base)']))
print(row_md('LoRA SFT (r=32, 67K data)',             results['LoRA SFT (checkpoint-1317)']))
print(row_md('LoRA SFT + DAPO',                       results['SFT + DAPO (ckpt-2760)'],      bold=True))
print(row_md('LoRA SFT + Dr. GRPO',                  results['SFT + Dr.GRPO (ckpt-1840)'],   bold=True))
print(row_md('SFT + DAPO + SC@8',                    results['SFT + DAPO + SC@8'],           bold=True))
print(row_md('Qwen2.5-1.5B-Instruct (official)',      results['Qwen2.5-1.5B-Instruct']))
print(row_md('Qwen2.5-Math-1.5B-Instruct (official)', results['Qwen2.5-Math-Instruct']))

| Method | GSM8K Accuracy | Avg Tokens | Truncated |
|--------|:-:|:-:|:-:|
| Zero-shot (Base) | 65.3% | 268 | 5.5% |
| CoT Prompting | 70.6% | 274 | 4.9% |
| LoRA SFT (r=32, 67K data) | 71.3% | 127 | 0.5% |
| **LoRA SFT + DAPO** | **78.2%** | **138** | **0.4%** |
| **LoRA SFT + Dr. GRPO** | **77.9%** | **139** | **0.4%** |
| **SFT + DAPO + SC@8** | **79.7%** | **—** | **—** |
| Qwen2.5-1.5B-Instruct (official) | 68.5% | 308 | 4.1% |
| Qwen2.5-Math-1.5B-Instruct (official) | 85.7% | 301 | 4.1% |
